# Práctica: Despliegue y uso de modelos en AI Foundry

Este repositorio contiene la práctica para aprender a desplegar y usar modelos en AI Foundry. La práctica está dividida en 3 partes independientes. Cada parte debe entregarse como un notebook Jupyter (`.ipynb`) con un nombre claro y apropiado.


## 1) Text, JSON y Guardrails
Objetivo: Desplegar un modelo y realizar tres tipos de interacciones:
### 1.1- Generar texto. 
Simple generación de texto con system prompt y user prompt.

⭐ Suma puntos crear un chat interactivo por CLI que persista la memoria a corto plazo.

[Documentación](https://learn.microsoft.com/en-us/azure/foundry/foundry-models/how-to/generate-responses?tabs=python)


In [ ]:
import os

from openai import OpenAI

# API Key de Azure OpenAI
# API Key de Azure OpenAI
api_key = os.getenv("Key")  # Asegúrate de configurar esta variable de entorno con tu clave API
base_url = os.getenv("base_url")  # Asegúrate de configurar esta variable de entorno con tu URL personalizada
project_endpoint = "https://pruebauno.services.ai.azure.com/api/projects/projuno"
base_url = project_endpoint.rstrip("/") + "/openai/v1"

try:
    client = OpenAI(
        base_url=base_url,
        api_key=api_key,
    )   

    response = client.responses.create(
        model="gpt-4o-mini",
        input="What are the top 3 benefits of cloud computing? Be concise.",
        max_output_tokens=500,
    )

    print("\n" + "="*60)
    print("✅ RESPUESTA DEL MODELO")
    print("="*60)
    print(f"\n📝 Respuesta:\n{response.output_text}")
    print(f"\n⚙️  Estado: {response.status}")
    print(f"📊 Tokens utilizados: {response.usage.output_tokens}")
    print("\n" + "="*60 + "\n")
except Exception as e:
    print("\n" + "❌"*30)
    print(f"ERROR: Fallo en la API")
    print(f"Detalles: {str(e)}")
    print("❌"*30 + "\n")


✅ RESPUESTA DEL MODELO

📝 Respuesta:
1. **Scalability**: Easily adjust resources to meet demand without significant infrastructure investment.

2. **Cost Efficiency**: Reduce IT costs by eliminating the need for physical hardware and allowing pay-as-you-go pricing models.

3. **Accessibility**: Enable remote access to data and applications from any device with an internet connection.

⚙️  Estado: completed
📊 Tokens utilizados: 67




### 1.2- Generar respuesta estructurada en formato JSON.
Generación de respuesta estructurada en JSON.

[Documentación](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/structured-outputs?tabs=python-secure%2Cdotnet-entra-id&pivots=programming-language-python)


In [ ]:
import os

from pydantic import BaseModel
from openai import OpenAI
import json

# API Key de Azure OpenAI
# API Key de Azure OpenAI
api_key = os.getenv("Key")  # Asegúrate de configurar esta variable de entorno con tu clave API
base_url = os.getenv("base_url")  # Asegúrate de configurar esta variable de entorno con tu URL personalizada
try:
    client = OpenAI(  
      base_url = base_url,  
      api_key=api_key,
    )

    class CalendarEvent(BaseModel):
        name: str
        date: str
        participants: list[str]
        equipo_Ganador: str

    completion = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Extract the event information."},
            {"role": "user", "content": "El Atleti va a ganar la Champions League y Mario que es del barça va a llorar."},
        ],
        response_format=CalendarEvent,
    )

    event = completion.choices[0].message.parsed

    print("\n" + "="*60)
    print("📋 EVENTO EXTRAÍDO (JSON ESTRUCTURADO)")
    print("="*60)
    print(f"\n🎯 Nombre del evento: {event.name}")
    print(f"📅 Fecha: {event.date}")
    print(f"👥 Participantes: {', '.join(event.participants)}")
    print(f"🏆 Equipo Ganador: {event.equipo_Ganador}")
    print("\n📦 JSON Completo:")
    print(json.dumps(json.loads(completion.model_dump_json()), indent=2))
    print("\n" + "="*60 + "\n")
except Exception as e:
    print("\n" + "❌"*30)
    print(f"ERROR: Fallo en la API")
    print(f"Detalles: {str(e)}")
    print("❌"*30 + "\n")


📋 EVENTO EXTRAÍDO (JSON ESTRUCTURADO)

🎯 Nombre del evento: Champions League Final
📅 Fecha: 2023-06-10
👥 Participantes: Atletico Madrid, Barcelona, Mario
🏆 Equipo Ganador: Atletico Madrid

📦 JSON Completo:
{
  "id": "chatcmpl-DU7feu9nZSw7HeEJpZorEj5FfDsXM",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\"name\":\"Champions League Final\",\"date\":\"2023-06-10\",\"participants\":[\"Atletico Madrid\",\"Barcelona\",\"Mario\"],\"equipo_Ganador\":\"Atletico Madrid\"}",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null,
        "parsed": {
          "name": "Champions League Final",
          "date": "2023-06-10",
          "participants": [
            "Atletico Madrid",
            "Barcelona",
            "Mario"
          ],
          "equipo_Ganador": "Atletico Madrid"
        }
      },
 

### 1.3- Implementar y demostrar Guardrails. 
Crear Guardrails para el modelo, documentar el proceso y hacer pruebas contra el modelo.

[Documentación](https://learn.microsoft.com/en-us/azure/foundry/guardrails/how-to-create-guardrails?tabs=python)

### Entregable: 
Notebook con código que muestre la llamada al endpoint del modelo para cada caso, ejemplos de prompts, validación del JSON recibido y una sección que muestre cómo se configuran y activan los Guardrails.

In [ ]:
# Guardrails testing with Azure credentials

import os

from openai import OpenAI

# API Key de Azure OpenAI
# API Key de Azure OpenAI
api_key = os.getenv("Key")  # Asegúrate de configurar esta variable de entorno con tu clave API
base_url = os.getenv("base_url")  # Asegúrate de configurar esta variable de entorno con tu URL personalizada
try:
    client = OpenAI(  
      base_url = base_url,  
      api_key=api_key,
    )

    # Test prompts for guardrails severity detection
    test_prompts = [
        "What is the capital of France?",
        "Explain how artificial intelligence works",
        "Tell me about cloud computing benefits"
    ]

    print("\n" + "="*60)
    print("🛡️  PRUEBA DE GUARDRAILS")
    print("="*60 + "\n")
    
    for idx, prompt in enumerate(test_prompts, 1):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=200
            )
            print(f"📌 Prueba #{idx}")
            print(f"❓ Entrada: {prompt}")
            print(f"✅ Respuesta permitida: Sí")
            print(f"💬 Respuesta: {response.choices[0].message.content[:100]}...")
            print(f"🔒 Severidad: Baja (contenido no filtrado)\n")
        except Exception as e:
            print(f"📌 Prueba #{idx}")
            print(f"❓ Entrada: {prompt}")
            print(f"❌ Respuesta permitida: No")
            print(f"🚫 Error/Severidad: {str(e)}\n")
            
    print("="*60 + "\n")
except Exception as e:
    print("\n" + "❌"*30)
    print(f"ERROR: Fallo en la API")
    print(f"Detalles: {str(e)}")
    print("❌"*30 + "\n")


🛡️  PRUEBA DE GUARDRAILS

📌 Prueba #1
❓ Entrada: What is the capital of France?
✅ Respuesta permitida: Sí
💬 Respuesta: The capital of France is Paris....
🔒 Severidad: Baja (contenido no filtrado)

📌 Prueba #2
❓ Entrada: Explain how artificial intelligence works
✅ Respuesta permitida: Sí
💬 Respuesta: Artificial intelligence (AI) encompasses a range of technologies and methodologies that enable machi...
🔒 Severidad: Baja (contenido no filtrado)

📌 Prueba #3
❓ Entrada: Tell me about cloud computing benefits
✅ Respuesta permitida: Sí
💬 Respuesta: Cloud computing offers numerous benefits for individuals and organizations alike. Here are some key ...
🔒 Severidad: Baja (contenido no filtrado)




---

## 2) Reasoning y Function Calling
Objetivo: Practicar con modelos razonadores y ver el function calling

### 2.1- Razonamiento
Desplegar un modelo razonador y parametrizar distintos grados de razonamiento (low, medium, high)

[Documentación](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/reasoning?tabs=csharp%2Cgpt-5)

In [ ]:
import os

from openai import OpenAI
import json

# API Key de Azure OpenAI
# API Key de Azure OpenAI
api_key = os.getenv("Key")  # Asegúrate de configurar esta variable de entorno con tu clave API
base_url = os.getenv("base_url")  # Asegúrate de configurar esta variable de entorno con tu URL personalizada

try:
    client = OpenAI(  
      base_url = base_url,  
      api_key=api_key,
    )

    response = client.chat.completions.create(
      model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": "What steps should I think about when writing my first Python API?"},
        ],
        max_completion_tokens = 5000

    )

    print("\n" + "="*60)
    print("🧠 RESPUESTA CON RAZONAMIENTO")
    print("="*60)
    print(f"\n📊 Detalles de la respuesta:")
    print(f"   • Modelo: {response.model}")
    print(f"   • Tokens utilizados: {response.usage.total_tokens}")
    print(f"   • Tokens de entrada: {response.usage.prompt_tokens}")
    print(f"   • Tokens de salida: {response.usage.completion_tokens}")
    print(f"\n💡 Contenido:")
    data = json.loads(response.model_dump_json())
    print(json.dumps(data, indent=2, ensure_ascii=False))
    print("\n" + "="*60 + "\n")
except Exception as e:
    print("\n" + "❌"*30)
    print(f"ERROR: Fallo en la API")
    print(f"Detalles: {str(e)}")
    print("❌"*30 + "\n")


🧠 RESPUESTA CON RAZONAMIENTO

📊 Detalles de la respuesta:
   • Modelo: gpt-4o-mini-2024-07-18
   • Tokens utilizados: 887
   • Tokens de entrada: 20
   • Tokens de salida: 867

💡 Contenido:
{
  "id": "chatcmpl-DU7fpIKKJJX74Oi2e1IIYC6lWwZTB",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Creating your first Python API can be an exciting journey! Here’s a step-by-step guide to help you through the process:\n\n### 1. Determine the Purpose of Your API\n- **Identify the functionality**: What data will your API provide, and what operations will it support?\n- **Understand your audience**: Who will use your API, and what features will be most valuable to them?\n\n### 2. Choose the Framework\n- **Select a Python framework**: Popular options include:\n  - **Flask**: A lightweight and flexible framework for building web applications.\n  - **Django**: A full-featured web framework that includes an ORM and othe

In [ ]:
from openai import OpenAI
import json

# API Key de Azure OpenAI
api_key = os.getenv("Key")  # Asegúrate de configurar esta variable de entorno con tu clave API
base_url = os.getenv("base_url")  # Asegúrate de configurar esta variable de entorno con tu URL personalizada

try:
    client = OpenAI(
        api_key=api_key,
        base_url=base_url,
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": "What steps should I think about when writing my first Python API?"},
        ],
        max_tokens=5000
    )

    print("\n" + "="*60)
    print("📚 RESPUESTA RAZONAMIENTO - NIVEL BASE")
    print("="*60)
    print(f"\n🤖 Modelo: {response.model}")
    print(f"⏱️  Tokens: {response.usage.total_tokens} (entrada: {response.usage.prompt_tokens}, salida: {response.usage.completion_tokens})")
    print(f"\n📄 Respuesta JSON:")
    data = json.loads(response.model_dump_json())
    print(json.dumps(data, indent=2, ensure_ascii=False))
    print("\n" + "="*60 + "\n")
except Exception as e:
    print("\n" + "❌"*30)
    print(f"ERROR: Fallo en la API")
    print(f"Detalles: {str(e)}")
    print("❌"*30 + "\n")


📚 RESPUESTA RAZONAMIENTO - NIVEL BASE

🤖 Modelo: gpt-4o-mini-2024-07-18
⏱️  Tokens: 832 (entrada: 20, salida: 812)

📄 Respuesta JSON:
{
  "id": "chatcmpl-DU7g5yFknxG9apItGxK0n09ulzwfH",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Creating your first Python API can be an exciting project! Here are key steps to consider throughout the process:\n\n### 1. Define Your API Purpose\n- **Determine functionality**: Identify what your API will do, such as providing data, performing actions, or integrating with other services.\n- **Identify target audience**: Know who will use your API and what their needs are.\n\n### 2. Choose the Right Framework\n- **Flask**: Lightweight and easy for beginners.\n- **Django REST Framework**: Robust for building more complex applications.\n- **FastAPI**: Modern and high-performance, ideal for building APIs quickly.\n\n### 3. Set Up Your Development Environment\n- **Install P

In [ ]:
import os

from openai import OpenAI
import json

# API Key de Azure OpenAI
api_key = os.getenv("Key")  # Asegúrate de configurar esta variable de entorno con tu clave API
base_url = os.getenv("base_url")  # Asegúrate de configurar esta variable de entorno con tu URL personalizada

try:
    client = OpenAI(
        base_url=base_url,
        api_key=api_key
    )

    # Test with different levels of reasoning (if supported by the model)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that provides detailed explanations."},
            {"role": "user", "content": "What steps should I think about when writing my first Python API?"},
        ],
        max_tokens=1000,
    )

    print("\n" + "="*60)
    print("🧠 RESPUESTA RAZONAMIENTO - NIVEL MEDIO")
    print("="*60)
    print(f"\n🤖 Modelo: {response.model}")
    print(f"⏱️  Tokens: {response.usage.total_tokens} (entrada: {response.usage.prompt_tokens}, salida: {response.usage.completion_tokens})")
    print(f"\n📄 Respuesta JSON:")
    data = json.loads(response.model_dump_json())
    print(json.dumps(data, indent=2, ensure_ascii=False))
    print("\n" + "="*60 + "\n")
except Exception as e:
    print("\n" + "❌"*30)
    print(f"ERROR: Fallo en la API")
    print(f"Detalles: {str(e)}")
    print("❌"*30 + "\n")


🧠 RESPUESTA RAZONAMIENTO - NIVEL MEDIO

🤖 Modelo: gpt-4o-mini-2024-07-18
⏱️  Tokens: 902 (entrada: 34, salida: 868)

📄 Respuesta JSON:
{
  "id": "chatcmpl-DU7gHFSJrWrT8W3WQIbgx1RCqXNXj",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Creating your first Python API can be an exciting project! Here’s a step-by-step guide to help you through the process:\n\n### 1. Define the Purpose and Scope of the API\n- **Identify Use Cases:** Determine what functionality your API will provide. What problem is it solving? Who is the target audience?\n- **Plan Endpoints:** Outline the endpoints you want to create. Consider what resources and actions your API will expose (e.g., CRUD operations).\n\n### 2. Choose the Right Tools and Libraries\n- **Framework Selection:** Select a web framework. Popular choices include:\n  - **Flask:** Great for small to medium applications and easy to learn.\n  - **FastAPI:** Modern and 

In [ ]:
import os

from openai import OpenAI
import json

# API Key de Azure OpenAI
api_key = os.getenv("Key")  # Asegúrate de configurar esta variable de entorno con tu clave API
base_url = os.getenv("base_url")  # Asegúrate de configurar esta variable de entorno con tu URL personalizada

try:
    client = OpenAI(
        api_key=api_key,
        base_url=base_url,
    )

    # Test with different levels of reasoning (if supported by the model)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that provides detailed explanations about Python APIs."},
            {"role": "user", "content": "What steps should I think about when writing my first Python API?"},
        ],
        max_tokens=2000,
    )

    print("\n" + "="*60)
    print("🧠 RESPUESTA RAZONAMIENTO - NIVEL AVANZADO")
    print("="*60)
    print(f"\n🤖 Modelo: {response.model}")
    print(f"⏱️  Tokens: {response.usage.total_tokens} (entrada: {response.usage.prompt_tokens}, salida: {response.usage.completion_tokens})")
    print(f"\n📄 Respuesta JSON:")
    data = json.loads(response.model_dump_json())
    print(json.dumps(data, indent=2, ensure_ascii=False))
    print("\n" + "="*60 + "\n")
except Exception as e:
    print("\n" + "❌"*30)
    print(f"ERROR: Fallo en la API")
    print(f"Detalles: {str(e)}")
    print("❌"*30 + "\n")


🧠 RESPUESTA RAZONAMIENTO - NIVEL AVANZADO

🤖 Modelo: gpt-4o-mini-2024-07-18
⏱️  Tokens: 1041 (entrada: 37, salida: 1004)

📄 Respuesta JSON:
{
  "id": "chatcmpl-DU7gRIFkmQuRtgtZb2DesROcQ7Imm",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Creating your first Python API can be an exciting and rewarding experience! Below are the essential steps you should consider when developing your API:\n\n### 1. Define the Purpose and Requirements\n- **Determine the functionality:** What do you want your API to do? Define the core functionality and features you want to expose.\n- **Identify the target audience:** Who will use your API? Understanding the users will help you tailor the API to their needs.\n\n### 2. Design the API\n- **Choose an architectural style:** Most APIs are RESTful. Decide if you want to follow REST principles, GraphQL, or other models.\n- **Define endpoints:** Create a list of endpoints that 

### 2.2- Function calling
Activar un motor de búsqueda web para probar llamadas a funciones (`function calling`) que recuperen información externa.

[Documentación](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/web-search)

⭐ Suma puntos usar deep research o hacer function calling con una función custom.


### Entregable: 
Notebook que muestre:
	- Ejemplos comparativos (misma tarea con distintos niveles de razonamiento).
	- Integración del web search y ejemplo de `function calling` que combine resultados externos con la respuesta del modelo.

In [ ]:
from openai import OpenAI

# API Key de Azure OpenAI
api_key = os.getenv("Key")  # Asegúrate de configurar esta variable de entorno con tu clave API
base_url = os.getenv("base_url")  # Asegúrate de configurar esta variable

try:
    # Set up client if not already configured
    client = OpenAI(  
      base_url = base_url,  
      api_key=api_key,
    )

    # Use the client to perform web search
    response = client.responses.create(   
      model="gpt-4o-mini",
      tools=[{"type": "web_search"}], 
      input="Please perform a web search on the latest trends in renewable energy"
    )

    print("\n" + "="*60)
    print("🔍 BÚSQUEDA WEB CON FUNCTION CALLING")
    print("="*60)
    print(f"\n🌐 Herramienta: Web Search")
    print(f"📌 Consulta: Latest trends in renewable energy")
    print(f"\n💡 Resultado:")
    print(f"{response.output_text}")
    print("\n" + "="*60 + "\n")
except Exception as e:
    print("\n" + "❌"*30)
    print(f"ERROR: Fallo en la API")
    print(f"Detalles: {str(e)}")
    print("❌"*30 + "\n")


🔍 BÚSQUEDA WEB CON FUNCTION CALLING

🌐 Herramienta: Web Search
📌 Consulta: Latest trends in renewable energy

💡 Resultado:
Here are the latest trends in renewable energy for 2026, reflecting advancements and shifts in technology, policy, and market dynamics:

1. **Energy Storage Innovations**:
   - The deployment of next-generation lithium-silicon and sodium-ion batteries is expanding significantly, along with grid-scale storage solutions. This is crucial for balancing intermittent solar and wind energy, making 24/7 clean energy more feasible [Renewable Energy Blog](https://www.renewableenergyconference.org/blog/trends-in-renewable-energy-2026.html).

2. **Green Hydrogen Adoption**:
   - Green hydrogen is evolving from a niche to a mainstream energy solution. National hydrogen strategies are increasingly being launched globally, with investments leading to lower electrolyzer costs. Hydrogen is particularly poised to decarbonize sectors where direct electrification is challenging [Star

In [ ]:
import os

from openai import OpenAI

# API Key de Azure OpenAI
api_key = os.getenv("Key")  # Asegúrate de configurar esta variable de entorno con tu clave API
base_url = os.getenv("base_url")  # Asegúrate de configurar esta variable

try:
    client = OpenAI(  
      base_url = base_url,  
      api_key=api_key,
    )

    response = client.responses.create(   
      model="gpt-4o-mini",
      tools=[{"type": "web_search"}], 
      input="Please perform a web search on the latest trends in renewable energy"
    )

    print("\n" + "="*60)
    print("🔍 BÚSQUEDA WEB CON FUNCTION CALLING")
    print("="*60)
    print(f"\n🌐 Herramienta: Web Search")
    print(f"📌 Consulta: Latest trends in renewable energy")
    print(f"\n💡 Resultado:")
    print(f"{response.output_text}")
    print("\n" + "="*60 + "\n")
except Exception as e:
    print("\n" + "❌"*30)
    print(f"ERROR: Fallo en la API")
    print(f"Detalles: {str(e)}")
    print("❌"*30 + "\n")


🔍 BÚSQUEDA WEB CON FUNCTION CALLING

🌐 Herramienta: Web Search
📌 Consulta: Latest trends in renewable energy

💡 Resultado:
Here are some of the latest trends in renewable energy as of 2026:

1. **Energy Storage Innovations**: Advances in battery technology, such as lithium-silicon and sodium-ion batteries, are crucial for integrating renewable energy sources like wind and solar. Long-duration energy storage systems are also gaining traction, enabling 24/7 access to renewable energy, thereby addressing the issue of intermittency [StartUs Insights](https://www.startus-insights.com/innovators-guide/renewable-energy-trends/).

2. **Solar Power Advances**: The deployment of perovskite solar cells, which promise higher efficiency and lower production costs, is revolutionizing solar technology. Additionally, floating solar farms are expanding in regions with limited land but ample water bodies, while Building-Integrated Photovoltaics (BIPV) are increasingly being adopted in urban infrastruct

---

## 3) Modelos Multimodales
### Objetivo: 
Desplegar un modelo multimodal y probar interacciones que involucren imágenes, audio y/o texto combinado (por ejemplo: describir una imagen, transcribir audio y responder preguntas sobre su contenido, etc.).

[Documentación](https://learn.microsoft.com/en-us/azure/foundry-classic/foundry-models/how-to/use-chat-multi-modal?context=%2Fazure%2Ffoundry%2Fcontext%2Fcontext&pivots=programming-language-python)

### Entregable: 
Notebook con llamadas al endpoint multimodal mostrando varios ejemplos: subida/consulta de imágenes, audios y prompts mixtos; incluir control de formatos y manejo de respuestas (texto y/o estructuras).

---

Formato y criterios de entrega
- Cada parte debe entregarse como un notebook `.ipynb` autocontenido que incluya:
	- Sección de configuración / credenciales (explicando cómo configurar variables de entorno localmente).
	- Código reproducible conecta a un modelo ya desplegado, realiza las llamadas y procesa las respuestas.
	- Celdas de explicación y resultados visibles (salidas, figuras, JSON validados).
	- Una sección final de conclusiones y problemas encontrados.

In [ ]:
# Multimodal demo (sin audio): describir imagen y responder pregunta
# Requiere: requests, base64

import base64
import requests
from openai import OpenAI
import os

# Intentamos usar un captioner local como fallback cuando el endpoint remoto no procesa imágenes.
# Para el captioner local, usamos la utilidad implementada en `describe_image.py` (BLIP).
try:
    import describe_image
    _has_local_caption = True
except Exception:
    _has_local_caption = False

# API Key de Azure OpenAI
api_key = "os.getenv('Key')"  # Asegúrate de configurar esta variable de entorno con tu clave API
base_url = os.getenv("base_url")  # Asegúrate de configurar esta variable de entorno con tu URL personalizada

client = OpenAI(base_url=base_url, api_key=api_key)


def describe_image_from_url(image_url: str) -> str:
    """Descarga la imagen y primero intenta que el endpoint remoto la describa.
    Si el endpoint indica que no puede procesar imágenes/URLs o falla, usamos
    el captioner local de respaldo (si está disponible).
    """
    try:
        resp = requests.get(image_url, timeout=15)
        resp.raise_for_status()
        content_type = resp.headers.get("Content-Type", "image/jpeg")
        b64 = base64.b64encode(resp.content).decode('utf-8')
        data_uri = f"data:{content_type};base64,{b64}"

        prompt = (
            "A continuación te proporciono una imagen en formato data URI. "
            "Por favor, describe detalladamente lo que aparece en la imagen, incluyendo objetos, contexto y cualquier detalle relevante.\n\n"
            f"IMAGEN: {data_uri}\n\n"
        )

        try:
            out = client.responses.create(model="gpt-4o-mini", input=prompt, max_output_tokens=800)
            text = getattr(out, 'output_text', str(out))
            lower = text.lower() if isinstance(text, str) else ''
            # Si el modelo remoto indica que no puede procesar imágenes/URLs, caer al captioner local
            if (('no puedo' in lower or 'cannot' in lower or "can't" in lower or 'no puedo analizar' in lower)
                and ('imagen' in lower or 'image' in lower or 'url' in lower or 'imagenes' in lower or 'imágenes' in lower)):
                if _has_local_caption:
                    return describe_image._local_caption_from_bytes(resp.content)
                return "El endpoint remoto no procesa imágenes y no hay captioner local disponible."
            return text
        except Exception as e_remote:
            # Fallback: si hay captioner local, usarlo
            if _has_local_caption:
                return describe_image._local_caption_from_bytes(resp.content)
            return f"ERROR describiendo imagen (remoto falló): {e_remote}"

    except Exception as e:
        return f"ERROR describiendo imagen: {e}"


def multimodal_qa(image_url: str, question: str):
    """Describe la imagen y responde la pregunta usando sólo la descripción."""
    print('\n' + '='*60)
    print('🔬 MULTIMODAL DEMO: imagen + pregunta (sin audio)')
    print('='*60 + '\n')

    print('1) Describiendo imagen...')
    img_desc = describe_image_from_url(image_url)
    print('\n--- DESCRIPCIÓN DE LA IMAGEN ---')
    print(img_desc)

    print('\n2) Pregunta al modelo basada en la descripción...')
    combined_prompt = (
        "Tengo la siguiente descripción de una imagen. "
        "Usa esta descripción para responder claramente la pregunta. Incluye referencias a la fuente (imagen) cuando sea relevante.\n\n"
        f"DESCRIPCIÓN IMAGEN:\n{img_desc}\n\n"
        f"PREGUNTA: {question}\n\n"
    )

    try:
        answer = client.responses.create(
            model="gpt-4o-mini",
            input=combined_prompt,
            max_output_tokens=800,
        )
        print('\n--- RESPUESTA MULTIMODAL ---')
        if hasattr(answer, 'output_text'):
            print(answer.output_text)
        else:
            print(str(answer))
    except Exception as e:
        print('\n❌ Error llamando al modelo para la pregunta:', e)
        print('\n--- RESPUESTA MULTIMODAL (USANDO SOLO LA DESCRIPCIÓN LOCAL) ---')
        print('Basado en la descripción, respuesta sugerida:')
        # Una respuesta simple basada en la descripción si el modelo falla
        print(f"La imagen contiene: {img_desc}\nPregunta: {question}\nRespuesta: Basado en la descripción anterior, parece que ...")

    print('\n' + '='*60 + '\n')

# Ejemplo de uso con la imagen proporcionada
image_url = "https://news.microsoft.com/source/wp-content/uploads/2024/04/The-Phi-3-small-language-models-with-big-potential-1-1900x1069.jpg"
question = "Describe la imagen y menciona los elementos más destacados."

multimodal_qa(image_url, question)

print('Multimodal demo lista.')



🔬 MULTIMODAL DEMO: imagen + pregunta (sin audio)

1) Describiendo imagen...


c:\Users\Alumno_AI\Downloads\githubalvfoundry\practicaFoundry\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 473/473 [00:00<00:00, 17532.33it/s]
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- DESCRIPCIÓN DE LA IMAGEN ---
a graph with a line graph and a line graph with a line graph on it

2) Pregunta al modelo basada en la descripción...

--- RESPUESTA MULTIMODAL ---
La imagen muestra un gráfico que presenta múltiples líneas en un mismo plano. Los elementos más destacados son las diferentes líneas del gráfico, que probablemente representan series de datos variando a lo largo del tiempo o en función de otra variable. Sin más detalles específicos sobre los ejes o la leyenda del gráfico, es difícil proporcionar un análisis más profundo. Sin embargo, la superposición de las líneas sugiere que puede haber comparaciones significativas entre los conjuntos de datos representados.


Multimodal demo lista.
